# M5/T1 Homework: Агент с памятью, новыми tool'ами и защитой от поломок

В этой работе я использую самописного ReAct-агента с историей сообщений.
Такой вариант проще всего анализировать: видны схема tool'ов, цикл вызовов,
история диалога и место, где можно встроить защиту от нежелательного поведения.

## 1. Постановка задачи

Нужно выполнить четыре части задания из `task.md`:
- добавить 3 новых tool'а и проверить их отдельно;
- протестировать вопросы, требующие нескольких tool-вызовов;
- намеренно сломать агента и затем починить его;
- написать короткий отчёт по результатам.

В качестве сценария поломки я выбрал зацикливание агента: сначала покажу, как
базовый агент бесполезно повторяет один и тот же вызов tool'а, затем добавлю
защиту от повторяющегося паттерна.

In [1]:
import datetime as dt
import json
import math
import os
import random
import re
from collections import Counter
from collections.abc import Callable
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI

SEED = 42
random.seed(SEED)

load_dotenv(".env")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY was not found in the repository root .env file or environment.")

MODEL = "openai/gpt-4o-mini"


def make_client() -> OpenAI:
    """Create an OpenAI-compatible client pointed to OpenRouter."""
    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )


client = make_client()
print("Model:", MODEL)

Model: openai/gpt-4o-mini


## 2. Базовые и новые tool'ы

Базовые tool'ы беру из семинара:
- `calculator`
- `get_current_time`
- `count_words`

Три новых tool'а для домашней работы:
- `extract_numbers` извлекает числа из текста;
- `sum_numbers` суммирует список чисел;
- `date_plus_days` двигает дату на заданное число дней и возвращает день недели.

`extract_numbers -> sum_numbers -> calculator` даёт естественный пример
последовательной цепочки вызовов.

In [2]:
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression with access to the math module."""
    try:
        allowed = {name: getattr(math, name) for name in dir(math) if not name.startswith("_")}
        allowed["math"] = math
        return str(eval(expression, {"__builtins__": {}}, allowed))
    except Exception as exc:
        return f"Error: {exc}"


def get_current_time() -> str:
    """Return the current UTC date and time."""
    return dt.datetime.now(dt.UTC).strftime("%Y-%m-%d %H:%M:%S UTC")


def count_words(text: str) -> str:
    """Count words and characters in the provided text."""
    words = text.split()
    return f"Words: {len(words)}, Characters: {len(text)}"


def extract_numbers(text: str) -> str:
    """Extract integers and floats from text and return them as JSON."""
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    numbers: list[float | int] = []
    for value in matches:
        number = float(value)
        numbers.append(int(number) if number.is_integer() else number)
    return json.dumps(numbers, ensure_ascii=True)


def sum_numbers(numbers: list[float]) -> str:
    """Sum a list of numbers and return the result as string."""
    return str(sum(numbers))


def date_plus_days(date_str: str, days: int) -> str:
    """Shift an ISO date by N days and return the new date plus weekday."""
    date_obj = dt.date.fromisoformat(date_str)
    new_date = date_obj + dt.timedelta(days=days)
    return f"{new_date.isoformat()} ({new_date.strftime('%A')})"


TOOL_MAP: dict[str, Callable[..., str]] = {
    "calculator": calculator,
    "get_current_time": get_current_time,
    "count_words": count_words,
    "extract_numbers": extract_numbers,
    "sum_numbers": sum_numbers,
    "date_plus_days": date_plus_days,
}

TOOLS_SCHEMA: list[dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a Python math expression such as math.sqrt(144) or 2**10.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current UTC date and time.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "count_words",
            "description": "Count words and characters in a text string.",
            "parameters": {
                "type": "object",
                "properties": {"text": {"type": "string"}},
                "required": ["text"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "extract_numbers",
            "description": "Extract all integers and floats from text and return them as a JSON array.",
            "parameters": {
                "type": "object",
                "properties": {"text": {"type": "string"}},
                "required": ["text"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "sum_numbers",
            "description": "Sum a list of numeric values.",
            "parameters": {
                "type": "object",
                "properties": {
                    "numbers": {
                        "type": "array",
                        "items": {"type": "number"},
                    }
                },
                "required": ["numbers"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "date_plus_days",
            "description": "Shift an ISO date by the given number of days and return date plus weekday.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date_str": {"type": "string", "description": "Date in YYYY-MM-DD format."},
                    "days": {"type": "integer"},
                },
                "required": ["date_str", "days"],
            },
        },
    },
]

print("Registered tools:", [tool["function"]["name"] for tool in TOOLS_SCHEMA])

Registered tools: ['calculator', 'get_current_time', 'count_words', 'extract_numbers', 'sum_numbers', 'date_plus_days']


## 3. Быстрая локальная проверка новых tool'ов

Это дешёвый офлайн smoke test до обращения к модели.

In [3]:
tool_smoke_tests = {
    "extract_numbers": extract_numbers("Revenue: 12.5, costs: 7, delta: -3"),
    "sum_numbers": sum_numbers([10, 20, 3.5]),
    "date_plus_days": date_plus_days("2026-04-07", 10),
}
tool_smoke_tests

{'extract_numbers': '[12.5, 7, -3]',
 'sum_numbers': '33.5',
 'date_plus_days': '2026-04-17 (Friday)'}

## 4. Самописный агент с историей сообщений

Агент хранит весь диалог в `self.messages`, умеет выполнять tool calls и
считает статистику использования инструментов.

In [4]:
SYSTEM_PROMPT = """
You are a careful ReAct agent with access to tools.
Use tools when real computation or transformation is needed.
For independent subtasks you may call several tools in one turn.
For dependent subtasks, chain tools step by step.
Keep final answers concise and include the intermediate computed facts.
""".strip()


class ReActAgent:
    """Minimal agent with persistent chat history and tool execution."""

    def __init__(
        self,
        client: Any,
        system_prompt: str,
        tools_schema: list[dict[str, Any]],
        tool_map: dict[str, Callable[..., str]],
        model: str = MODEL,
        max_steps: int = 8,
    ) -> None:
        self.client = client
        self.system_prompt = system_prompt
        self.tools_schema = tools_schema
        self.tool_map = tool_map
        self.model = model
        self.max_steps = max_steps
        self.tool_usage: Counter[str] = Counter()
        self.messages: list[dict[str, Any]] = [{"role": "system", "content": system_prompt}]

    def reset(self) -> None:
        self.messages = [{"role": "system", "content": self.system_prompt}]
        self.tool_usage.clear()

    def _model_call(self) -> Any:
        return self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            tools=self.tools_schema,
            tool_choice="auto",
        )

    def execute_tool(self, tool_call: Any) -> str:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        if name not in self.tool_map:
            return f"Error: unknown tool '{name}'"
        result = str(self.tool_map[name](**args))
        self.tool_usage[name] += 1
        return result

    def chat(self, user_message: str, verbose: bool = True) -> str:
        self.messages.append({"role": "user", "content": user_message})

        for step in range(1, self.max_steps + 1):
            response = self._model_call()
            msg = response.choices[0].message

            if not msg.tool_calls:
                final_text = msg.content or ""
                self.messages.append({"role": "assistant", "content": final_text})
                if verbose:
                    print(f"[Final answer at step {step}] {final_text}")
                return final_text

            assistant_payload = {
                "role": "assistant",
                "content": msg.content,
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in msg.tool_calls
                ],
            }
            self.messages.append(assistant_payload)

            for tc in msg.tool_calls:
                result = self.execute_tool(tc)
                if verbose:
                    print(f"[Step {step}] {tc.function.name}({tc.function.arguments}) -> {result}")
                self.messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

        return "Reached max steps without a final answer."

    def tool_report(self) -> dict[str, int]:
        return dict(self.tool_usage)


agent = ReActAgent(
    client=client,
    system_prompt=SYSTEM_PROMPT,
    tools_schema=TOOLS_SCHEMA,
    tool_map=TOOL_MAP,
)

## 5. Live-тесты на несколько tool'ов

Ниже две проверочные задачи из условия:
- параллельные вызовы, где подзадачи независимы;
- последовательные вызовы, где следующий tool зависит от результата предыдущего.

Эти клетки рассчитаны на запуск с реальным OpenRouter API ключом.

In [5]:
parallel_prompt = """
Solve three independent tasks:
1. Tell me the current UTC time.
2. Count words in the text 'Large language models need careful evaluation'.
3. Compute sqrt(2025).
Use tools where appropriate and then give one final answer.
""".strip()

print(parallel_prompt)

Solve three independent tasks:
1. Tell me the current UTC time.
2. Count words in the text 'Large language models need careful evaluation'.
3. Compute sqrt(2025).
Use tools where appropriate and then give one final answer.


In [6]:
sequential_prompt = """
Take the text 'The batch contained 12, 18 and 45 units'.
First extract all numbers from it.
Then sum those numbers.
Then compute the square root of the sum.
Also tell me what date will be 14 days after 2026-04-07.
Use tools and explain the chain briefly in the final answer.
""".strip()

print(sequential_prompt)

Take the text 'The batch contained 12, 18 and 45 units'.
First extract all numbers from it.
Then sum those numbers.
Then compute the square root of the sum.
Also tell me what date will be 14 days after 2026-04-07.
Use tools and explain the chain briefly in the final answer.


In [7]:
agent.reset()
parallel_answer = agent.chat(parallel_prompt, verbose=True)
print(parallel_answer)
print(agent.tool_report())

[Step 1] get_current_time({}) -> 2026-04-07 08:44:15 UTC
[Step 1] count_words({"text": "Large language models need careful evaluation"}) -> Words: 6, Characters: 45
[Step 1] calculator({"expression": "sqrt(2025)"}) -> 45.0
[Final answer at step 2] Here are the results for the tasks:

1. **Current UTC time**: 2026-04-07 08:44:15 UTC
2. **Word count**: 6 words
3. **Square root of 2025**: 45.0

Final answer: Current UTC time is 2026-04-07 08:44:15 UTC, the word count is 6, and the square root of 2025 is 45.0.
Here are the results for the tasks:

1. **Current UTC time**: 2026-04-07 08:44:15 UTC
2. **Word count**: 6 words
3. **Square root of 2025**: 45.0

Final answer: Current UTC time is 2026-04-07 08:44:15 UTC, the word count is 6, and the square root of 2025 is 45.0.
{'get_current_time': 1, 'count_words': 1, 'calculator': 1}


In [8]:
agent.reset()
sequential_answer = agent.chat(sequential_prompt, verbose=True)
print(sequential_answer)
print(agent.tool_report())

[Step 1] extract_numbers({"text": "The batch contained 12, 18 and 45 units"}) -> [12, 18, 45]
[Step 1] date_plus_days({"date_str": "2026-04-07", "days": 14}) -> 2026-04-21 (Tuesday)
[Step 2] sum_numbers({"numbers":[12,18,45]}) -> 75
[Step 3] calculator({"expression":"math.sqrt(75)"}) -> 8.660254037844387
[Final answer at step 4] Here’s how I solved the problem:

1. **Extract Numbers**: From the text 'The batch contained 12, 18 and 45 units', I extracted the numbers [12, 18, 45].
2. **Sum the Numbers**: I summed these numbers to get 75.
3. **Compute Square Root**: The square root of 75 is approximately 8.66.
4. **Date Calculation**: I calculated that 14 days after 2026-04-07 is 2026-04-21 (Tuesday).

Final results:
- Sum of the numbers: 75
- Square root of the sum: approximately 8.66
- Date after 14 days: 2026-04-21 (Tuesday)
Here’s how I solved the problem:

1. **Extract Numbers**: From the text 'The batch contained 12, 18 and 45 units', I extracted the numbers [12, 18, 45].
2. **Sum t

## 6. Ломаем агента: искусственное зацикливание

Здесь я не завишу от сети: создаю поддельный клиент, который всё время просит
вызвать один и тот же tool. Базовый агент честно выполняет эти вызовы до
`max_steps`, то есть фактически зацикливается.

In [9]:
class FakeFunction:
    def __init__(self, name: str, arguments: str) -> None:
        self.name = name
        self.arguments = arguments


class FakeToolCall:
    def __init__(self, call_id: str, name: str, arguments: str) -> None:
        self.id = call_id
        self.function = FakeFunction(name=name, arguments=arguments)


class FakeMessage:
    def __init__(self, content: str | None = None, tool_calls: list[FakeToolCall] | None = None) -> None:
        self.content = content
        self.tool_calls = tool_calls or []


class FakeChoice:
    def __init__(self, message: FakeMessage) -> None:
        self.message = message


class FakeResponse:
    def __init__(self, message: FakeMessage) -> None:
        self.choices = [FakeChoice(message)]


class LoopingCompletions:
    def __init__(self, repeats: int = 10) -> None:
        self.repeats = repeats
        self.counter = 0

    def create(self, **_: Any) -> FakeResponse:
        self.counter += 1
        if self.counter <= self.repeats:
            call = FakeToolCall(
                call_id=f"loop-{self.counter}",
                name="get_current_time",
                arguments="{}",
            )
            return FakeResponse(FakeMessage(content=None, tool_calls=[call]))
        return FakeResponse(FakeMessage(content="Loop finished unexpectedly."))


class LoopingClient:
    def __init__(self, repeats: int = 10) -> None:
        self.chat = type("ChatNamespace", (), {"completions": LoopingCompletions(repeats=repeats)})()


loop_client = LoopingClient(repeats=10)
broken_agent = ReActAgent(
    client=loop_client,
    system_prompt=SYSTEM_PROMPT,
    tools_schema=TOOLS_SCHEMA,
    tool_map=TOOL_MAP,
    max_steps=4,
)

broken_result = broken_agent.chat("What time is it?", verbose=True)
print("Broken agent result:", broken_result)

[Step 1] get_current_time({}) -> 2026-04-07 08:44:27 UTC
[Step 2] get_current_time({}) -> 2026-04-07 08:44:27 UTC
[Step 3] get_current_time({}) -> 2026-04-07 08:44:27 UTC
[Step 4] get_current_time({}) -> 2026-04-07 08:44:27 UTC
Broken agent result: Reached max steps without a final answer.


## 7. Чиним агента: защита от повторяющихся tool-вызовов

Добавляю простую защиту: если агент несколько шагов подряд вызывает один и тот
же tool с теми же аргументами, выполнение останавливается с понятным сообщением.
Это не универсальное решение всех проблем, но оно дешёвое и хорошо ловит
типичный runaway-паттерн.

In [10]:
class SafeReActAgent(ReActAgent):
    """ReAct agent with simple repeated-tool-call protection."""

    def __init__(self, *args: Any, repeat_limit: int = 2, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.repeat_limit = repeat_limit

    def chat(self, user_message: str, verbose: bool = True) -> str:
        self.messages.append({"role": "user", "content": user_message})
        repeated_signatures: list[str] = []

        for step in range(1, self.max_steps + 1):
            response = self._model_call()
            msg = response.choices[0].message

            if not msg.tool_calls:
                final_text = msg.content or ""
                self.messages.append({"role": "assistant", "content": final_text})
                if verbose:
                    print(f"[Final answer at step {step}] {final_text}")
                return final_text

            signatures = [f"{tc.function.name}:{tc.function.arguments}" for tc in msg.tool_calls]
            repeated_signatures.extend(signatures)
            if len(repeated_signatures) >= self.repeat_limit:
                last_n = repeated_signatures[-self.repeat_limit :]
                if len(set(last_n)) == 1:
                    stop_message = (
                        "Stopped to prevent a loop: the same tool call was repeated "
                        f"{self.repeat_limit} times in a row ({last_n[-1]})."
                    )
                    self.messages.append({"role": "assistant", "content": stop_message})
                    if verbose:
                        print(stop_message)
                    return stop_message

            assistant_payload = {
                "role": "assistant",
                "content": msg.content,
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in msg.tool_calls
                ],
            }
            self.messages.append(assistant_payload)

            for tc in msg.tool_calls:
                result = self.execute_tool(tc)
                if verbose:
                    print(f"[Step {step}] {tc.function.name}({tc.function.arguments}) -> {result}")
                self.messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

        return "Reached max steps without a final answer."


safe_loop_client = LoopingClient(repeats=10)
safe_agent = SafeReActAgent(
    client=safe_loop_client,
    system_prompt=SYSTEM_PROMPT,
    tools_schema=TOOLS_SCHEMA,
    tool_map=TOOL_MAP,
    max_steps=8,
    repeat_limit=2,
)

safe_result = safe_agent.chat("What time is it?", verbose=True)
print("Safe agent result:", safe_result)

[Step 1] get_current_time({}) -> 2026-04-07 08:44:27 UTC
Stopped to prevent a loop: the same tool call was repeated 2 times in a row (get_current_time:{}).
Safe agent result: Stopped to prevent a loop: the same tool call was repeated 2 times in a row (get_current_time:{}).


## 8. Краткий отчёт

**Что сделано**
- Реализован самописный агент с историей сообщений и ручным выполнением tool calls.
- Добавлены 3 новых tool'а: `extract_numbers`, `sum_numbers`, `date_plus_days`.
- Подготовлены два тестовых вопроса:
  один на независимые подзадачи, другой на последовательную цепочку инструментов.
- Показана поломка через искусственное зацикливание и добавлена защита от
  повторяющегося вызова одного и того же tool'а.

**Наблюдения**
- Самописный агент удобен для обучения: легко видеть каждое решение модели и
  каждое сообщение в памяти.
- Последовательные цепочки особенно чувствительны к качеству схемы tool'ов:
  если schema и описания неясны, модель хуже строит plan.
- Простая защита по сигнатуре tool-вызова не решает все проблемы, но резко
  снижает риск бессмысленного расхода шагов и токенов.